# Daily Challenge: Building an agent with LangGraph and the Gemini API: **BaristaBot** ☕

A stateful, tool-augmented café ordering assistant built as a **LangGraph** graph on top of the **Gemini API** (via `langchain-google-genai`).
- **State:** `OrderState` (messages + order + finished flag), a `TypedDict`.
- **Nodes:** `chatbot` (LLM), `human` (user I/O), `tools` (auto-executed `get_menu`), `ordering` (stateful order tools).
- **Edges:** conditional routing chatbot → {tools, ordering, human, END}, human → {chatbot, END}; loops until the order is placed.

> **How to run.**
> - **With Gemini (recommended):** add your key from [AI Studio](https://aistudio.google.com/app/apikey) as a **Kaggle secret** or a **Colab secret** named `GOOGLE_API_KEY` (🔑 icon in the Colab sidebar, enable notebook access), or set the `GOOGLE_API_KEY` environment variable.
> - **Interactive:** set `INTERACTIVE = True` below to chat yourself (`q` to quit).
>
> This saved copy was executed in an environment **without a Gemini key**. In that case the notebook automatically falls back to a **local tool-calling model**, `Qwen/Qwen2.5-7B-Instruct` in 4-bit GGUF run on CPU with llama.cpp, wrapped as a LangChain chat model, so the *same* LangGraph code runs end to end. A scripted customer plays the human so that the outputs can be saved. The graph, state, tools and routing code are identical with Gemini; only the `llm` object changes.

## 1. Install and import LangGraph and LangChain support for the Gemini API

In [ ]:
%pip install -qU "langgraph==1.0.5" "langchain-google-genai==4.1.2" "google-genai==1.56.0" "langgraph-prebuilt==1.0.5"
# only needed for the no-API-key fallback:
%pip install -q "transformers<5" llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

Note: `langgraph-prebuilt` is pinned too. With `-U`, pip otherwise pulls a newer `langgraph-prebuilt` (1.0.13) that imports `ExecutionInfo` from a newer `langgraph`, which breaks `from langgraph.prebuilt import ToolNode` with langgraph 1.0.5.

## 2. Set up your API key

In [ ]:
import os
os.environ["USE_TF"] = "0"

GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    try:                                        # Kaggle (as in the course)
        from kaggle_secrets import UserSecretsClient
        GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    except Exception:
        pass
if not GOOGLE_API_KEY:
    try:                                        # Google Colab
        from google.colab import userdata
        GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
    except Exception:
        pass
if GOOGLE_API_KEY:
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
USE_GEMINI = bool(GOOGLE_API_KEY)

INTERACTIVE = False      # True -> you type the customer's messages with input()
print("Gemini API key found:", USE_GEMINI, "| interactive mode:", INTERACTIVE)

## 3. Define core instructions: the state and the system instruction

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph.message import add_messages


class OrderState(TypedDict):
    """State representing the customer's order conversation."""

    # The chat conversation. This preserves the conversation history
    # between nodes. The `add_messages` annotation indicates to LangGraph
    # that state is updated by appending returned messages, not replacing
    # them.
    messages: Annotated[list, add_messages]

    # The customer's in-progress order.
    order: list[str]

    # Flag indicating that the order is placed and completed.
    finished: bool


# The system instruction defines how the chatbot is expected to behave and includes
# rules for when to call different functions, as well as rules for the conversation, such
# as tone and what is permitted for discussion.
BARISTABOT_SYSINT = (
    "system",  # 'system' indicates the message is a system instruction.
    "You are a BaristaBot, an interactive cafe ordering system. A human will talk to you about the "
    "available products you have and you will answer any questions about menu items (and only about "
    "menu items - no off-topic discussion, but you can chat about the products and their history). "
    "The customer will place an order for 1 or more items from the menu, which you will structure "
    "and send to the ordering system after confirming the order with the human. "
    "\n\n"
    "Add items to the customer's order with add_to_order, and reset the order with clear_order. "
    "To see the contents of the order so far, call get_order (this is shown to you, not the user) "
    "Always confirm_order with the user (double-check) before calling place_order. Calling confirm_order will "
    "display the order items to the user and returns their response to seeing the list. Their response may contain modifications. "
    "Always verify and respond with drink and modifier names from the MENU before adding them to the order. "
    "If you are unsure a drink or modifier matches those on the MENU, ask a question to clarify or redirect. "
    "You only have the modifiers listed on the menu. "
    "Once the customer has finished ordering items, Call confirm_order to ensure it is correct then make "
    "any necessary updates and then call place_order. Once place_order has returned, thank the user and "
    "say goodbye!",
)

# This is the message with which the system opens the conversation.
WELCOME_MSG = "Welcome to the BaristaBot cafe. Type `q` to quit. How may I serve you today?"

### The LLM: Gemini, or the local fallback
The local fallback is a minimal LangChain `BaseChatModel`. It converts LangChain messages to the Qwen chat template, passes the bound tools' JSON schemas, and parses `<tool_call>{...}</tool_call>` blocks back into `AIMessage.tool_calls`. That is exactly the interface LangGraph's `ToolNode` and our routing functions expect.

In [ ]:
import json, re, uuid
from typing import Any, List, Optional, Sequence
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.utils.function_calling import convert_to_openai_tool

_CACHE = {}

class LocalToolChat(BaseChatModel):
    model_id: str = "Qwen/Qwen2.5-1.5B-Instruct"      # tokenizer / chat template (and weights if no gguf given)
    gguf_repo: str = ""                                # optional: run a quantized GGUF with llama.cpp instead
    gguf_file: str = ""
    max_new_tokens: int = 256
    tool_schemas: list = []

    @property
    def _llm_type(self): return "local-tool-chat"

    def _load(self):
        # weights are cached at module level so that bound copies (bind_tools) share one model in memory
        key = (self.model_id, self.gguf_file)
        if key not in _CACHE:
            tok = AutoTokenizer.from_pretrained(self.model_id)
            if self.gguf_file:
                from llama_cpp import Llama
                mdl = Llama.from_pretrained(repo_id=self.gguf_repo, filename=self.gguf_file, n_ctx=8192, verbose=False)
            else:
                mdl = AutoModelForCausalLM.from_pretrained(self.model_id, torch_dtype=torch.float32).eval()
            _CACHE[key] = (tok, mdl)
        return _CACHE[key]

    def bind_tools(self, tools: Sequence[Any], **kw):
        return self.model_copy(update={"tool_schemas": [convert_to_openai_tool(t) for t in tools]})

    def _to_chat(self, messages):
        out = []
        for m in messages:
            if isinstance(m, SystemMessage): out.append({"role": "system", "content": m.content})
            elif isinstance(m, HumanMessage): out.append({"role": "user", "content": m.content})
            elif isinstance(m, ToolMessage): out.append({"role": "tool", "content": str(m.content)})
            elif isinstance(m, AIMessage):
                d = {"role": "assistant", "content": m.content or ""}
                if m.tool_calls:
                    d["tool_calls"] = [{"type": "function", "function": {"name": tc["name"], "arguments": tc["args"]}} for tc in m.tool_calls]
                out.append(d)
        return out

    def _generate(self, messages: List[BaseMessage], stop=None, run_manager=None, **kw):
        tok, mdl = self._load()
        chat = self._to_chat(messages)
        if self.gguf_file:
            prompt = tok.apply_chat_template(chat, tools=self.tool_schemas or None, add_generation_prompt=True, tokenize=False)
            text = mdl.create_completion(prompt, max_tokens=self.max_new_tokens, temperature=0, stop=["<|im_end|>"])["choices"][0]["text"].strip()
        else:
            ids = tok.apply_chat_template(chat, tools=self.tool_schemas or None, add_generation_prompt=True, return_tensors="pt")
            with torch.no_grad():
                out = mdl.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=self.max_new_tokens, do_sample=False)
            text = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()
        calls = []
        for blob in re.findall(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", text, flags=re.S):
            try:
                j = json.loads(blob); calls.append({"name": j["name"], "args": j.get("arguments", {}) or {}, "id": "call_" + uuid.uuid4().hex[:8], "type": "tool_call"})
            except Exception: pass
        content = re.sub(r"<tool_call>.*?</tool_call>", "", text, flags=re.S).strip()
        return ChatResult(generations=[ChatGeneration(message=AIMessage(content=content, tool_calls=calls))])


if USE_GEMINI:
    from langchain_google_genai import ChatGoogleGenerativeAI
    # The exercise suggests "gemini-2.5-flash-latest"; the stable model id is "gemini-2.5-flash".
    llm = ChatGoogleGenerativeAI(model=os.environ.get("GEMINI_MODEL", "gemini-2.5-flash"), temperature=0)
else:
    # Qwen2.5-7B-Instruct, 4-bit GGUF run by llama.cpp on CPU (a 1.5B model was tried first and barely called the tools)
    llm = LocalToolChat(model_id="Qwen/Qwen2.5-7B-Instruct", gguf_repo="bartowski/Qwen2.5-7B-Instruct-GGUF",
                        gguf_file="Qwen2.5-7B-Instruct-Q4_K_M.gguf", max_new_tokens=200)
print("LLM backend:", type(llm).__name__)

In [ ]:
# Customer input: real input() when interactive, otherwise a scripted customer (so the notebook can run unattended)
from collections import deque
SCRIPT = deque()

def ask_user(prompt):
    if INTERACTIVE:
        return input(prompt)
    reply = SCRIPT.popleft() if SCRIPT else "q"
    print(prompt + reply)
    return reply

## 4. Define a single-turn chatbot

In [ ]:
from langgraph.graph import StateGraph, START, END

def chatbot(state: OrderState) -> OrderState:
    """The chatbot itself. A simple wrapper around the model's own chat interface."""
    message_history = [BARISTABOT_SYSINT] + state["messages"]
    return {"messages": [llm.invoke(message_history)]}


# Set up the initial graph based on our state definition.
graph_builder = StateGraph(OrderState)

# Add the chatbot function to the app graph as a node called "chatbot".
graph_builder.add_node("chatbot", chatbot)

# Define the chatbot node as the app entrypoint.
graph_builder.add_edge(START, "chatbot")

chat_graph = graph_builder.compile()

## 5. Visualise the graph

In [ ]:
from IPython.display import Image, display, Markdown

def show_graph(graph):
    try:
        display(Image(graph.get_graph().draw_mermaid_png()))     # rendered via the mermaid.ink web service
    except Exception as e:
        print(f"(PNG rendering unavailable: {type(e).__name__}); Mermaid source:")
        print(graph.get_graph().draw_mermaid())

show_graph(chat_graph)

## 6. Run the graph once

In [ ]:
from pprint import pprint

user_msg = "Hello, what can you do?"
state = chat_graph.invoke({"messages": [user_msg]})

# The state object contains lots of information. Uncomment the pprint lines to see it all.
# pprint(state)

# Note that the final state now has 2 messages. Our HumanMessage, and an additional AIMessage.
for msg in state["messages"]:
    print(f"{type(msg).__name__}: {msg.content}")

## 7. A second conversational turn

In [ ]:
user_msg = "Oh great, what kinds of latte can you make?"

state["messages"].append(user_msg)
state = chat_graph.invoke(state)

# pprint(state)
for msg in state["messages"]:
    print(f"{type(msg).__name__}: {msg.content}")

Without a menu, the model can only **guess** which lattes the café sells (hallucination risk). The live-menu tool in step 10 fixes this.

## 8. Add a human node

In [ ]:
from langchain_core.messages.ai import AIMessage


def human_node(state: OrderState) -> OrderState:
    """Display the last model message to the user, and receive the user's input."""
    last_msg = state["messages"][-1]
    print("Model:", last_msg.content)

    user_input = ask_user("User: ")

    # If it looks like the user is trying to quit, flag the conversation
    # as over.
    if user_input in {"q", "quit", "exit", "goodbye"}:
        state["finished"] = True

    return state | {"messages": [("user", user_input)]}


def chatbot_with_welcome_msg(state: OrderState) -> OrderState:
    """The chatbot itself. A wrapper around the model's own chat interface."""

    if state["messages"]:
        # If there are messages, continue the conversation with the Gemini model.
        new_output = llm.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        # If there are no messages, start with the welcome message.
        new_output = AIMessage(content=WELCOME_MSG)

    return state | {"messages": [new_output]}


# Start building a new graph.
graph_builder = StateGraph(OrderState)

# Add the chatbot and human nodes to the app graph.
graph_builder.add_node("chatbot", chatbot_with_welcome_msg)
graph_builder.add_node("human", human_node)

# Start with the chatbot again.
graph_builder.add_edge(START, "chatbot")

# The chatbot will always go to the human next.
graph_builder.add_edge("chatbot", "human");

## 9. Conditional edge: exit the loop

In [ ]:
from typing import Literal


def maybe_exit_human_node(state: OrderState) -> Literal["chatbot", "__end__"]:
    """Route to the chatbot, unless it looks like the user is exiting."""
    if state.get("finished", False):
        return END
    else:
        return "chatbot"


graph_builder.add_conditional_edges("human", maybe_exit_human_node)

chat_with_human_graph = graph_builder.compile()

show_graph(chat_with_human_graph)

In [ ]:
# Try the chat loop (scripted customer here; set INTERACTIVE = True to type yourself)
SCRIPT.extend(["Hi! Do you serve coffee?", "q"])
state = chat_with_human_graph.invoke({"messages": []})
print("\nfinished flag:", state.get("finished"))

## 10. Add a live menu tool

In [ ]:
from langchain_core.tools import tool


@tool
def get_menu() -> str:
    """Provide the latest up-to-date menu."""
    # Note that this is just hard-coded text, but you could connect this to a live stock
    # database, or you could use Gemini's multi-modal capabilities and take live photos of
    # your cafe's chalk menu or the products on the counter and assmble them into an input.

    return """
    MENU:
    Coffee Drinks:
    Espresso
    Americano
    Cold Brew

    Coffee Drinks with Milk:
    Latte
    Cappuccino
    Cortado
    Macchiato
    Mocha
    Flat White

    Tea Drinks:
    English Breakfast Tea
    Green Tea
    Earl Grey

    Tea Drinks with Milk:
    Chai Latte
    Matcha Latte
    London Fog

    Other Drinks:
    Steamer
    Hot Chocolate

    Modifiers:
    Milk options: Whole, 2%, Oat, Almond, 2% Lactose Free; Default option: whole
    Espresso shots: Single, Double, Triple, Quadruple; default: Double
    Caffeine: Decaf, Regular; default: Regular
    Hot-Iced: Hot, Iced; Default: Hot
    Sweeteners (option to add one or more): vanilla sweetener, hazelnut sweetener, caramel sauce, chocolate sauce, sugar free vanilla sweetener
    Special requests: any reasonable modification that does not involve items not on the menu, for example: 'extra hot', 'one pump', 'half caff', 'extra foam', etc.

    "dirty" means add a shot of espresso to a drink that doesn't usually have it, like "Dirty Chai Latte".
    "Regular milk" is the same as 'whole milk'.
    "Sweetened" means add some regular sugar, not a sweetener.

    Soy milk has run out of stock today, so soy is not available.
  """

In [ ]:
from langgraph.prebuilt import ToolNode


# Define the tools and create a "tools" node.
tools = [get_menu]
tool_node = ToolNode(tools)

# Attach the tools to the model so that it knows what it can call.
llm_with_tools = llm.bind_tools(tools)


def maybe_route_to_tools(state: OrderState) -> Literal["tools", "human"]:
    """Route between human or tool nodes, depending if a tool call is made."""
    if not (msgs := state.get("messages", [])):
        raise ValueError(f"No messages found when parsing state: {state}")

    # Only route based on the last message.
    msg = msgs[-1]

    # When the chatbot returns tool_calls, route to the "tools" node.
    if hasattr(msg, "tool_calls") and len(msg.tool_calls) > 0:
        return "tools"
    else:
        return "human"


def chatbot_with_tools(state: OrderState) -> OrderState:
    """The chatbot with tools. A simple wrapper around the model's own chat interface."""
    defaults = {"order": [], "finished": False}

    if state["messages"]:
        new_output = llm_with_tools.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        new_output = AIMessage(content=WELCOME_MSG)

    # Set up some defaults if not already set, then pass through the provided state,
    # overriding only the "messages" field.
    return defaults | state | {"messages": [new_output]}


graph_builder = StateGraph(OrderState)

# Add the nodes, including the new tool_node.
graph_builder.add_node("chatbot", chatbot_with_tools)
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", tool_node)

# Chatbot may go to tools, or human.
graph_builder.add_conditional_edges("chatbot", maybe_route_to_tools)
# Human may go back to chatbot, or exit.
graph_builder.add_conditional_edges("human", maybe_exit_human_node)

# Tools always route back to chat afterwards.
graph_builder.add_edge("tools", "chatbot")

graph_builder.add_edge(START, "chatbot")
graph_with_menu = graph_builder.compile()

show_graph(graph_with_menu)

In [ ]:
SCRIPT.extend(["What teas do you have on the menu?", "q"])
state = graph_with_menu.invoke({"messages": []})
print("\nTool messages in the conversation:", [m.name for m in state["messages"] if type(m).__name__ == "ToolMessage"])

## 11. Handle orders

In [ ]:
from collections.abc import Iterable
from random import randint

from langgraph.prebuilt import InjectedState
from langchain_core.messages.tool import ToolMessage

# These functions have no body; LangGraph does not allow @tools to update
# the conversation state, so you will implement a separate node to handle
# state updates. Using @tools is still very convenient for defining the tool
# schema, so empty functions have been defined that will be bound to the LLM
# but their implementation is deferred to the order_node.


@tool
def add_to_order(drink: str, modifiers: Iterable[str]) -> str:
    """Adds the specified drink to the customer's order, including any modifiers.

    Returns:
      The updated order in progress.
    """


@tool
def confirm_order() -> str:
    """Asks the customer if the order is correct.

    Returns:
      The user's free-text response.
    """


@tool
def get_order() -> str:
    """Returns the users order so far. One item per line."""


@tool
def clear_order():
    """Removes all items from the user's order."""


@tool
def place_order() -> int:
    """Sends the order to the barista for fulfillment.

    Returns:
      The estimated number of minutes until the order is ready.
    """


def order_node(state: OrderState) -> OrderState:
    """The ordering node. This is where the order state is manipulated."""
    tool_msg = state.get("messages", [])[-1]
    order = state.get("order", [])
    outbound_msgs = []
    order_placed = False

    for tool_call in tool_msg.tool_calls:

        if tool_call["name"] == "add_to_order":

            # Each order item is just a string. This is where it assembled as "drink (modifiers, ...)".
            modifiers = tool_call["args"].get("modifiers", [])
            if isinstance(modifiers, str):                     # small models sometimes send a single string
                modifiers = [modifiers]
            modifier_str = ", ".join(modifiers) if modifiers else "no modifiers"

            order.append(f'{tool_call["args"]["drink"]} ({modifier_str})')
            response = "\n".join(order)

        elif tool_call["name"] == "confirm_order":

            # We could entrust the LLM to do order confirmation, but it is a good practice to
            # show the user the exact data that comprises their order so that what they confirm
            # precisely matches the order that goes to the kitchen - avoiding hallucination
            # or reality skew.

            # In a real scenario, this is where you would connect your POS screen to show the
            # order to the user.

            print("Your order:")
            if not order:
                print("  (no items)")

            for drink in order:
                print(f"  {drink}")

            response = ask_user("Is this correct? ")

        elif tool_call["name"] == "get_order":

            response = "\n".join(order) if order else "(no order)"

        elif tool_call["name"] == "clear_order":

            order.clear()
            response = "Order cleared."

        elif tool_call["name"] == "place_order":

            order_text = "\n".join(order)
            print("Sending order to kitchen!")
            print(order_text)

            # TODO(you!): Implement cafe.
            order_placed = True
            response = randint(1, 5)  # ETA in minutes

        else:
            raise NotImplementedError(f'Unknown tool call: {tool_call["name"]}')

        # Record the tool results as tool messages.
        outbound_msgs.append(
            ToolMessage(
                content=str(response),
                name=tool_call["name"],
                tool_call_id=tool_call["id"],
            )
        )

    return {"messages": outbound_msgs, "order": order, "finished": order_placed}


def maybe_route_to_tools(state: OrderState) -> str:
    """Route between chat and tool nodes if a tool call is made."""
    if not (msgs := state.get("messages", [])):
        raise ValueError(f"No messages found when parsing state: {state}")

    msg = msgs[-1]

    if state.get("finished", False):
        # When an order is placed, exit the app. The system instruction indicates
        # that the chatbot should say thanks and goodbye at this point, so we can exit
        # cleanly.
        return END

    elif hasattr(msg, "tool_calls") and len(msg.tool_calls) > 0:
        # Route to `tools` node for any automated tool calls first.
        if any(
            tool["name"] in tool_node.tools_by_name.keys() for tool in msg.tool_calls
        ):
            return "tools"
        else:
            return "ordering"

    else:
        return "human"

## 12. Define the complete graph

In [ ]:
# Auto-tools will be invoked automatically by the ToolNode
auto_tools = [get_menu]
tool_node = ToolNode(auto_tools)

# Order-tools will be handled by the order node.
order_tools = [add_to_order, confirm_order, get_order, clear_order, place_order]

# The LLM needs to know about all of the tools, so specify everything here.
llm_with_tools = llm.bind_tools(auto_tools + order_tools)


graph_builder = StateGraph(OrderState)

# Nodes
graph_builder.add_node("chatbot", chatbot_with_tools)
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", tool_node)
graph_builder.add_node("ordering", order_node)

# Chatbot -> {ordering, tools, human, END}
graph_builder.add_conditional_edges("chatbot", maybe_route_to_tools)
# Human -> {chatbot, END}
graph_builder.add_conditional_edges("human", maybe_exit_human_node)

# Tools (both kinds) always route back to chat afterwards.
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge("ordering", "chatbot")

graph_builder.add_edge(START, "chatbot")
graph_with_order_tools = graph_builder.compile()

show_graph(graph_with_order_tools)

### Run the complete ordering system
The scripted customer asks about the menu, asks which teas are from England, orders two drinks, **modifies** the order, confirms, and the graph exits on its own after `place_order`.

In [ ]:
# The default recursion limit for traversing nodes is 25 - setting it higher
# means you can try a more complex order with multiple steps and round-trips.
config = {"recursion_limit": 100}

SCRIPT.clear()
SCRIPT.extend(CUSTOMER_SCRIPT := [
    "Hi! What's on the menu today?",
    "Which teas are from England?",
    "I'd like an iced latte with oat milk and vanilla sweetener, please.",
    "Also an Earl Grey tea. That's everything.",
    "Actually, please make the latte with almond milk instead of oat. Otherwise it's correct.",
    "Yes, that's correct now.",
    "Yes, please place it.",
])

state = graph_with_order_tools.invoke({"messages": []}, config)

# Things to try:
# - Order a drink!
# - Make a change to your order.
# - "Which teas are from England?"
# - Note that the graph should naturally exit after placing an order.

pprint(state)

In [ ]:
# Readable transcript + what reached the "kitchen"
for m in state["messages"]:
    kind = type(m).__name__
    if kind == "AIMessage" and m.tool_calls:
        print(f"🤖 tool calls -> {[(tc['name'], tc['args']) for tc in m.tool_calls]}")
    elif kind == "ToolMessage":
        print(f"   🔧 {m.name} returned: {m.content[:120]!r}")
    else:
        print(f"{'👤' if kind == 'HumanMessage' else '🤖'} {m.content}")
print("\nFINAL ORDER:", state.get("order"), "| finished:", state.get("finished"))

ANALYSIS